# 03 · Entrenamiento y evaluación del modelo predictivo

Este notebook documenta, celda a celda, la lógica real de `ml/modelo_prediccion.py` (Árbol de Decisión sobre hurtos agregados por mes×municipio×tipo de hurto), para que el resultado sea reproducible y quede como evidencia para el jurado.

> Requiere las variables de entorno de Supabase (`SUPABASE_HOST`, `SUPABASE_PORT`, `SUPABASE_DB`, `SUPABASE_USER`, `SUPABASE_PASSWORD`) configuradas en un archivo `.env` en la raíz del proyecto.

## 1. Conexión y extracción de datos históricos
Se agregan los registros por **mes, municipio y tipo de hurto** (no se usa `armas_medios` en el modelo).

In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import psycopg2
from dotenv import load_dotenv
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
load_dotenv()

conn = psycopg2.connect(
    host=os.getenv("SUPABASE_HOST"),
    port=int(os.getenv("SUPABASE_PORT", 5432)),
    dbname=os.getenv("SUPABASE_DB", "postgres"),
    user=os.getenv("SUPABASE_USER"),
    password=os.getenv("SUPABASE_PASSWORD"),
    sslmode="require",
)

In [ ]:
query = """
    SELECT
        u.municipio, u.departamento, u.id AS ubicacion_id,
        th.id AS tipo_hurto_id, th.codigo AS tipo_hurto_codigo,
        DATE_TRUNC('month', rh.fecha_hecho)::DATE AS mes,
        SUM(rh.cantidad) AS total_hurtos
    FROM registros_hurto rh
    JOIN ubicaciones u  ON u.id  = rh.ubicacion_id
    JOIN tipos_hurto th ON th.id = rh.tipo_hurto_id
    GROUP BY u.municipio, u.departamento, u.id, th.id, th.codigo,
             DATE_TRUNC('month', rh.fecha_hecho)
    ORDER BY mes, u.municipio
"""
df = pd.read_sql(query, conn)
print(f"{len(df):,} filas | {df['municipio'].nunique()} municipios | {df['mes'].min()} → {df['mes'].max()}")
df.head()

## 2. Feature engineering
Variables reales usadas por el modelo: `mes_num`, `año`, `trimestre`, `depto_enc`, `tipo_enc`, `promedio_historico`, `lag_1` (mes anterior) y `lag_3` (3 meses atrás).

In [ ]:
df = df.copy()
df["mes"] = pd.to_datetime(df["mes"])
df["mes_num"]   = df["mes"].dt.month
df["año"]       = df["mes"].dt.year
df["trimestre"] = df["mes"].dt.quarter
df["total_hurtos"] = df["total_hurtos"].astype(float)

promedio = df.groupby(["ubicacion_id","tipo_hurto_id"])["total_hurtos"].mean().reset_index()
promedio.columns = ["ubicacion_id","tipo_hurto_id","promedio_historico"]
df = df.merge(promedio, on=["ubicacion_id","tipo_hurto_id"], how="left")

df = df.sort_values(["ubicacion_id","tipo_hurto_id","mes"])
df["lag_1"] = df.groupby(["ubicacion_id","tipo_hurto_id"])["total_hurtos"].shift(1)
df["lag_3"] = df.groupby(["ubicacion_id","tipo_hurto_id"])["total_hurtos"].shift(3)

le_depto = LabelEncoder(); le_tipo = LabelEncoder()
df["depto_enc"] = le_depto.fit_transform(df["departamento"])
df["tipo_enc"]  = le_tipo.fit_transform(df["tipo_hurto_codigo"])

df = df.dropna(subset=["lag_1","lag_3"])
df.shape

## 3. Split de entrenamiento/prueba

**Nota metodológica (documentada en `docs/conclusiones.md`):** el split actual es **aleatorio** (80/20), no cronológico. Para un problema de series de tiempo esto puede sobreestimar el desempeño real, porque el entrenamiento puede incluir meses posteriores a los del conjunto de prueba. Se deja así porque es lo que corre hoy en producción (`ml/modelo_prediccion.py`); la celda comentada más abajo muestra la alternativa cronológica recomendada como mejora futura.

In [ ]:
features = ["mes_num","año","trimestre","depto_enc","tipo_enc","promedio_historico","lag_1","lag_3"]
X = df[features]
y = df["total_hurtos"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
# --- Alternativa recomendada (NO usada en producción, solo referencia) ---
# df_sorted = df.sort_values("mes")
# corte = df_sorted["mes"].quantile(0.8)
# train = df_sorted[df_sorted["mes"] <= corte]
# test  = df_sorted[df_sorted["mes"] > corte]
# X_train, y_train = train[features], train["total_hurtos"]
# X_test,  y_test  = test[features],  test["total_hurtos"]

## 4. Entrenamiento del Árbol de Decisión

In [ ]:
modelo = DecisionTreeRegressor(max_depth=8, min_samples_leaf=10, random_state=42)
modelo.fit(X_train, y_train)

## 5. Evaluación: MAE y R²
**Copia estos dos valores a la sección 2 de `docs/conclusiones.md`.**

In [ ]:
y_pred = modelo.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f"MAE : {mae:.2f}  (error promedio en hurtos/mes)")
print(f"R²  : {r2:.4f}  (capacidad explicativa del modelo)")

## 6. Interpretabilidad: variables más influyentes
Respalda el punto de "el modelo no es una caja negra" citado en `docs/marco_metodologico.md`.

In [ ]:
importancias = pd.Series(modelo.feature_importances_, index=features).sort_values(ascending=False)
importancias

## 7. Lógica de negocio aplicada sobre la predicción (no es parte del modelo de ML)

Estas reglas se aplican **después** de la predicción del árbol, en `ml/modelo_prediccion.py`, y también deben documentarse como parte del sistema:

- **Nivel de riesgo:** por cuantiles históricos del municipio — `CRITICO` si ≥ 1.5× p75, `ALTO` si ≥ p75, `MEDIO` si ≥ p50, `BAJO` en el resto.
- **Confianza (`confianza_pct`):** heurística de cercanía entre la predicción y el promedio histórico del municipio+tipo, acotada entre 50 % y 95 % — no es un intervalo estadístico del modelo.
- **Horizonte de 3 meses:** los meses 2 y 3 reutilizan los mismos `lag_1`/`lag_3` del último dato real (no se recalculan iterativamente), por lo que se espera mayor error a medida que el horizonte avanza.

## 8. Resultado final para `docs/conclusiones.md`

> Completa con los valores reales obtenidos en la celda de evaluación:
> - MAE: `___`
> - R²: `___`
> - Variables más influyentes: `___`